# 01 · Tensors, shapes and dtypes

**Why this comes first.** In the coding test almost nothing you write will be *conceptually* hard. What costs points is a tensor with shape `(n,)` where `(n, 1)` was needed, or `float64` where `float32` was needed — and PyTorch often **does not raise an error**, it just silently computes the wrong thing. With no IDE and no autocomplete, the only debugger you have is your own ability to *predict* shapes and dtypes before running a cell.

**How to work through every notebook in this folder**

1. For "predict" exercises, write your prediction **before** running anything. Being wrong is the point — that is where the learning is.
2. `check(...)` cells give you ✅ / ❌. A ❌ with an error message in brackets usually means "not done yet".
3. Hints are folded (`▶ Hint`). Try for a few minutes before opening them.
4. Solutions are in `SOLUTIONS/`. Open them only after a real attempt, and afterwards **retype** the answer from memory rather than copying.

---
## Theory in 60 seconds

| Thing | Convention used in this course |
|---|---|
| Input matrix `X` | shape `(n_samples, n_features)`, dtype `float32` |
| Regression target | shape `(n_samples, 1)`, dtype `float32` — same shape as the model output |
| Binary target (BCE) | shape `(n_samples, 1)`, dtype `float32` (a "true probability" 0.0 or 1.0) |
| Multi-class target (CE) | shape `(n_samples,)`, dtype `int64` (a class **index**) |

**Broadcasting rule** (NumPy and PyTorch are identical): line the shapes up **from the right**; two dimensions are compatible if they are equal or one of them is 1; missing dimensions on the left count as 1.

```
(4, 1)        (4, 3)        (4, 1)
(   3)        (   3)        (   4)      <- a (4,) vector lines up with the LAST axis
------        ------        ------
(4, 3)        (4, 3)        (4, 4)      <- the classic silent bug
```

`axis` (NumPy) and `dim` (PyTorch) name the axis that **disappears** in a reduction: `x.sum(axis=0)` of a `(n, d)` array collapses the rows and leaves `(d,)` — one value per column.

In [1]:
import sys; sys.path.insert(0, "..")          # so the SOLUTIONS folder also finds dnd_utils
import numpy as np
import torch
import torch.nn as nn
from dnd_utils import check

---
## Exercise 1.1 · Predict the shape

`a` has shape `(5, 3)`. For each expression write the shape you expect **as a tuple** (a scalar is `()`), then run the check.

In [2]:
a = np.arange(15, dtype=np.float32).reshape(5, 3)

pred = {
    "a[0]":                           (3,),      # an integer index removes that axis
    "a[:, 0]":                        (5,),      # same: the column axis is gone
    "a[:, [0]]":                      (5, 1),    # a list index keeps the axis (advanced indexing, makes a copy)
    "a[:, 0, None]":                  (5, 1),    # None inserts a new axis of length 1 (a view)
    "a[1:3, :2]":                     (2, 2),    # slices keep axes
    "a.sum()":                        (),        # everything collapsed -> scalar
    "a.sum(axis=0)":                  (3,),      # axis 0 (rows) disappears -> one value per column
    "a.mean(axis=1)":                 (5,),      # axis 1 (columns) disappears -> one value per row
    "a.mean(axis=1, keepdims=True)":  (5, 1),    # kept as length-1 axis -> broadcasts back against a
    "a.T @ a":                        (3, 3),    # (3,5) @ (5,3)
    "a @ a.T":                        (5, 5),    # (5,3) @ (3,5)
    "a[a[:, 0] > 5]":                 (3, 3),    # first column is 0,3,6,9,12 -> 3 rows survive
    "a.argmax(axis=1)":               (5,),      # index of the largest column, per row
    "a.reshape(-1)":                  (15,),
    "a.reshape(-1, 1)":               (15, 1),
}

In [3]:
for expr, p in pred.items():
    check(f"{expr:32s} -> {np.shape(eval(expr))}", lambda: tuple(p) == np.shape(eval(expr)))

✅ a[0]                             -> (3,)
✅ a[:, 0]                          -> (5,)
✅ a[:, [0]]                        -> (5, 1)
✅ a[:, 0, None]                    -> (5, 1)
✅ a[1:3, :2]                       -> (2, 2)
✅ a.sum()                          -> ()
✅ a.sum(axis=0)                    -> (3,)
✅ a.mean(axis=1)                   -> (5,)
✅ a.mean(axis=1, keepdims=True)    -> (5, 1)
✅ a.T @ a                          -> (3, 3)
✅ a @ a.T                          -> (5, 5)
✅ a[a[:, 0] > 5]                   -> (3, 3)
✅ a.argmax(axis=1)                 -> (5,)
✅ a.reshape(-1)                    -> (15,)
✅ a.reshape(-1, 1)                 -> (15, 1)


---
## Exercise 1.2 · The silent bug: `(n, 1)` against `(n,)`

A model outputs predictions of shape `(4, 1)`. The labels come out of a CSV as shape `(4,)`.

**a)** Before running: what is the shape of `p - y`? What would the "MSE" computed from it mean?

In [4]:
p = torch.tensor([[1.0], [2.0], [3.0], [4.0]])     # model output, (4, 1)
y = torch.tensor([1.0, 2.0, 3.0, 4.0])             # labels, (4,)   <- predictions are PERFECT

In [5]:
shape_of_difference = (4, 4)      # (4,1) vs (4,) -> every prediction is compared with EVERY label
# (p - y)[i, j] = p_i - y_j.  Squared differences: |i-j|=1 six times (1), |i-j|=2 four times (4), |i-j|=3 twice (9), 0 four times
wrong_mse_value = (6 * 1 + 4 * 4 + 2 * 9) / 16      # = 2.5, although the predictions are perfect

In [6]:
check("shape of p - y", lambda: tuple(shape_of_difference) == tuple((p - y).shape))
check("value of the wrong MSE", lambda: abs(wrong_mse_value - ((p - y) ** 2).mean().item()) < 1e-6)
print("nn.MSELoss gives the same wrong number (plus only a UserWarning):", nn.MSELoss()(p, y).item())

✅ shape of p - y
✅ value of the wrong MSE
nn.MSELoss gives the same wrong number (plus only a UserWarning): 2.5


/usr/local/lib/python3.13/site-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([4])) that is different to the input size (torch.Size([4, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


**b)** Fix the label shape in **three different ways** so the MSE becomes `0`. You will need all three idioms sooner or later.

In [7]:
y_fix1 = y.reshape(-1, 1)
y_fix2 = y[:, None]
y_fix3 = y.unsqueeze(1)

In [8]:
for name, yy in (("reshape", y_fix1), ("None-indexing", y_fix2), ("unsqueeze", y_fix3)):
    check(f"{name}: shape (4, 1) and MSE 0", lambda: yy.shape == (4, 1) and nn.MSELoss()(p, yy).item() == 0.0)

✅ reshape: shape (4, 1) and MSE 0
✅ None-indexing: shape (4, 1) and MSE 0
✅ unsqueeze: shape (4, 1) and MSE 0


**c)** Think about it: if you *train* with the buggy `(n, n)` loss, what will the network learn to output? (You will reproduce this in notebook 06.)

**Answer.** The buggy loss is the mean over all pairs `(p_i - y_j)²`. For each prediction `p_i`, the sum over `j` of `(p_i - y_j)²` is minimised when `p_i` equals the **mean of all labels** in the batch. So the network learns to output (roughly) the same constant — the label mean — for every input, and the loss gets stuck at about the **variance of the labels**. Symptom to remember: *"my regression loss goes down a bit and then is flat, and all predictions are almost identical"* → check the label shape.

---
## Exercise 1.3 · dtypes

Predict each dtype as a string such as `"float32"`, `"float64"`, `"int64"`, `"bool"`.

In [9]:
pred_dtype = {
    "np.array([1, 2, 3])":                               "int64",
    "np.array([1., 2., 3.])":                            "float64",    # NumPy's default float is 64-bit
    "np.array([1, 2, 3]) / 2":                           "float64",    # true division always gives floats
    "np.array([1, 2, 3]) > 1":                           "bool",
    "torch.tensor([1, 2, 3])":                           "int64",
    "torch.tensor([1., 2., 3.])":                        "float32",    # PyTorch's default float is 32-bit ...
    "torch.tensor(np.array([1., 2., 3.]))":              "float64",    # ... but a NumPy array KEEPS its dtype!
    "torch.tensor(np.array([1., 2.]), dtype=torch.float32)": "float32",
    "torch.tensor([1., 2.]).argmax()":                   "int64",      # indices are int64 -> directly usable as CE labels
    "(torch.tensor([1., 2.]) > 1).float()":              "float32",
    "torch.tensor([0, 1, 1]) == torch.tensor([0, 1, 0])": "bool",
}

In [10]:
for expr, p in pred_dtype.items():
    actual = str(eval(expr).dtype).replace("torch.", "")
    check(f"{expr:55s} -> {actual}", lambda: p == actual)

✅ np.array([1, 2, 3])                                     -> int64
✅ np.array([1., 2., 3.])                                  -> float64
✅ np.array([1, 2, 3]) / 2                                 -> float64
✅ np.array([1, 2, 3]) > 1                                 -> bool
✅ torch.tensor([1, 2, 3])                                 -> int64
✅ torch.tensor([1., 2., 3.])                              -> float32
✅ torch.tensor(np.array([1., 2., 3.]))                    -> float64
✅ torch.tensor(np.array([1., 2.]), dtype=torch.float32)   -> float32
✅ torch.tensor([1., 2.]).argmax()                         -> int64
✅ (torch.tensor([1., 2.]) > 1).float()                    -> float32
✅ torch.tensor([0, 1, 1]) == torch.tensor([0, 1, 0])      -> bool


**Why it matters.** `nn.Linear` creates `float32` weights. Feed it a `float64` tensor (which is what you get from `torch.tensor(numpy_array)` if the array came from `np.array([...])` or a CSV parsed with `float()`), and you get a `RuntimeError`. Run the next cell and **read the error message** so that you recognise it in the test.

In [11]:
layer = nn.Linear(3, 2)
x64 = torch.tensor(np.ones((4, 3)))            # float64!
try:
    layer(x64)
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float


In [12]:
x_a = torch.tensor(np.ones((4, 3)), dtype=torch.float32)
x_b = x64.float()                 # same as x64.to(torch.float32)

In [13]:
check("x_a works", lambda: layer(x_a).shape == (4, 2))
check("x_b works", lambda: layer(x_b).shape == (4, 2))

✅ x_a works
✅ x_b works


True

---
## Exercise 1.4 · Vectorised column operations

The course **requires vectorised solutions** — no Python loop over samples. The dataset below has 6 samples and 3 features, with missing values.

Compute, without any loop:

- `n_missing`: number of NaNs per column, shape `(3,)`
- `col_mean`: mean of each column ignoring NaNs, shape `(3,)`
- `filled`: a copy of `data` where every NaN is replaced by **its own column's** mean
- `standardised`: `filled` with every column shifted to mean 0 and scaled to (population) standard deviation 1

<details><summary>Hint</summary>

`np.isnan`, `np.nanmean(..., axis=0)`, and the three-argument `np.where(mask, value_if_true, value_if_false)` — the second argument may be a `(3,)` vector, it broadcasts against the `(6, 3)` mask.
</details>

In [14]:
data = np.array([[1.0, 200.0, np.nan],
                 [2.0, np.nan, 0.5],
                 [3.0, 180.0, 0.1],
                 [np.nan, 220.0, 0.9],
                 [5.0, 240.0, np.nan],
                 [4.0, 160.0, 0.5]], dtype=np.float32)

n_missing = np.isnan(data).sum(axis=0)                    # True counts as 1; axis 0 (rows) disappears
col_mean = np.nanmean(data, axis=0)                       # (3,)
filled = np.where(np.isnan(data), col_mean, data)         # (6,3) mask, (3,) means broadcast along rows
standardised = (filled - filled.mean(axis=0)) / filled.std(axis=0)     # (6,3) - (3,) -> column-wise

In [15]:
check("n_missing", lambda: n_missing.tolist() == [1, 1, 2])
check("col_mean", lambda: np.allclose(col_mean, [3.0, 200.0, 0.5]))
check("filled has no NaN and keeps valid values", lambda: not np.isnan(filled).any() and filled[0, 0] == 1 and np.isclose(filled[3, 0], 3.0) and np.isclose(filled[0, 2], 0.5))
check("standardised: column means 0, stds 1", lambda: np.allclose(standardised.mean(axis=0), 0, atol=1e-6) and np.allclose(standardised.std(axis=0), 1, atol=1e-6))

✅ n_missing
✅ col_mean
✅ filled has no NaN and keeps valid values
✅ standardised: column means 0, stds 1


True

**Understanding check.** Why does `filled - filled.mean(axis=0)` subtract column-wise, but `filled - filled.mean(axis=1)` would raise an error here? And what would you write to subtract each **row's** mean?

**Answer.** `filled` is `(6, 3)`. `mean(axis=0)` is `(3,)`, which lines up with the last axis (the 3 columns) → fine. `mean(axis=1)` is `(6,)`, which also tries to line up with the **last** axis of length 3 → `6 ≠ 3`, error. (If the array were square, e.g. `(3, 3)`, there would be *no error* and the result would silently be wrong.) To subtract row means you need shape `(6, 1)`: `filled - filled.mean(axis=1, keepdims=True)`.

---
## Exercise 1.5 · One-hot encoding, three ways

`idx` holds a category index per sample. Build the one-hot matrix of shape `(7, 4)` in three different ways.

<details><summary>Hint</summary>

(1) index into an identity matrix; (2) compare `idx[:, None]` with `np.arange(4)`; (3) create zeros and assign `1` with a pair of integer index arrays `[rows, cols]`.
</details>

In [16]:
idx = np.array([2, 0, 3, 3, 1, 0, 2])
K = 4

onehot_a = np.eye(K, dtype=np.float32)[idx]                           # row idx[i] of the identity for each sample
onehot_b = (idx[:, None] == np.arange(K)[None, :]).astype(np.float32) # (7,1) == (1,4) -> (7,4) bool
onehot_c = np.zeros((len(idx), K), dtype=np.float32)
onehot_c[np.arange(len(idx)), idx] = 1.0                              # element (i, idx[i]) for every i

In [17]:
ref = np.array([[0,0,1,0],[1,0,0,0],[0,0,0,1],[0,0,0,1],[0,1,0,0],[1,0,0,0],[0,0,1,0]])
for name, oh in (("a", onehot_a), ("b", onehot_b), ("c", onehot_c)):
    check(f"onehot_{name}", lambda: oh.shape == (7, 4) and (oh == ref).all())
check("going back: argmax recovers idx", lambda: (onehot_a.argmax(axis=1) == idx).all())

✅ onehot_a
✅ onehot_b
✅ onehot_c
✅ going back: argmax recovers idx


True

---
## Exercise 1.6 · Labels from rules (masks and `np.where`)

`grades` has shape `(n, 3)`: three grades (0–20) per student. Create integer labels, vectorised:

- **0** if **at least two** of the three grades are below 10,
- otherwise **1** if the average grade is below 14,
- otherwise **2**.

Then compute `counts`: how many students fall in each class, shape `(3,)`.

In [18]:
rng = np.random.default_rng(0)
grades = rng.integers(4, 20, size=(200, 3)).astype(np.float32)

low = (grades < 10).sum(axis=1) >= 2             # (200,) bool: count the True values per row
avg = grades.mean(axis=1)                        # (200,)
labels = np.where(low, 0, np.where(avg < 14, 1, 2))      # nested where = vectorised if / elif / else
counts = np.bincount(labels, minlength=3)

In [19]:
_ref = np.array([0 if (g < 10).sum() >= 2 else (1 if g.mean() < 14 else 2) for g in grades])   # the slow loop, only to check
check("labels", lambda: labels.shape == (200,) and (labels == _ref).all())
check("counts", lambda: list(counts) == list(np.bincount(_ref, minlength=3)))

✅ labels
✅ counts


True

---
## Exercise 1.7 · From logits to predictions to accuracy

`logits` is what a 3-class network outputs for 6 samples. Compute with **PyTorch** functions:

- `pred_class`: predicted class per sample, shape `(6,)`
- `accuracy`: fraction of correct predictions, as a Python `float`
- `n_correct_per_class`: for each true class, how many of its samples were predicted correctly, shape `(3,)`

Watch the dtype: the mean of a `bool` tensor is not defined in PyTorch.

In [20]:
logits = torch.tensor([[2.0, 0.1, -1.0],
                       [0.3, 0.2, 0.9],
                       [-1.0, 3.0, 0.0],
                       [0.0, 0.1, 0.05],
                       [5.0, 4.9, 4.8],
                       [-2.0, -1.0, -3.0]])
y_true = torch.tensor([0, 2, 1, 2, 0, 0])

pred_class = logits.argmax(dim=1)                          # dim 1 (the classes) disappears -> (6,)
correct = pred_class == y_true                             # bool (6,)
accuracy = correct.float().mean().item()                   # bool -> float before the mean; .item() -> Python float
n_correct_per_class = torch.bincount(y_true[correct], minlength=3)   # true classes of the correct ones

In [21]:
check("pred_class", lambda: pred_class.tolist() == [0, 2, 1, 1, 0, 1])
check("accuracy is a Python float = 4/6", lambda: isinstance(accuracy, float) and abs(accuracy - 4 / 6) < 1e-6)
check("n_correct_per_class", lambda: n_correct_per_class.tolist() == [2, 1, 1])

✅ pred_class
✅ accuracy is a Python float = 4/6
✅ n_correct_per_class


True

---
## Exercise 1.8 · What a `Linear` layer really is

`nn.Linear(in_features, out_features)` stores a weight matrix and a bias vector and computes `x @ W.T + b`.

**a)** Predict the shapes of `layer.weight` and `layer.bias` for `nn.Linear(3, 5)`, and the output shape for an input batch of shape `(8, 3)`.

In [22]:
weight_shape = (5, 3)       # (out_features, in_features): one ROW of weights per output neuron
bias_shape = (5,)           # one bias per output neuron
output_shape = (8, 5)       # the batch axis is untouched; the last axis goes 3 -> 5

In [23]:
torch.manual_seed(0)
layer = nn.Linear(3, 5)
xb = torch.randn(8, 3)
check("weight shape", lambda: tuple(weight_shape) == tuple(layer.weight.shape))
check("bias shape", lambda: tuple(bias_shape) == tuple(layer.bias.shape))
check("output shape", lambda: tuple(output_shape) == tuple(layer(xb).shape))

✅ weight shape
✅ bias shape
✅ output shape


True

**b)** Reproduce the layer's output by hand using only `layer.weight`, `layer.bias`, `@` and `+`.

In [24]:
manual_out = xb @ layer.weight.T + layer.bias        # (8,3) @ (3,5) -> (8,5); bias (5,) broadcasts over the batch

In [25]:
check("manual forward equals layer(xb)", lambda: torch.allclose(manual_out, layer(xb), atol=1e-6))

✅ manual forward equals layer(xb)


True

**c)** An MLP has layers `Linear(35, 25) → ReLU → Linear(25, 25) → ReLU → Linear(25, 1)`. Compute the number of trainable parameters **on paper**, then verify.

In [26]:
n_params_by_hand = (35 * 25 + 25) + (25 * 25 + 25) + (25 * 1 + 1)      # weights + biases per layer = 900 + 650 + 26 = 1576

In [27]:
mlp = nn.Sequential(nn.Linear(35, 25), nn.ReLU(), nn.Linear(25, 25), nn.ReLU(), nn.Linear(25, 1))
check("parameter count", lambda: n_params_by_hand == sum(p.numel() for p in mlp.parameters()))

✅ parameter count


True

---
## Exercise 1.9 · Getting values out of tensors

Tensors that are part of a computation graph (anything computed from model parameters) cannot be converted to NumPy directly. Predict which of these lines raise an error, then run the cell.

In [28]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
out = (w * 3).sum()

attempts = {
    "out.item()":            lambda: out.item(),              # 1
    "out.numpy()":           lambda: out.numpy(),             # 2
    "out.detach().numpy()":  lambda: out.detach().numpy(),    # 3
    "float(out)":            lambda: float(out),              # 4
    "(w * 3).tolist()":      lambda: (w * 3).tolist(),        # 5
    "(w * 3).item()":        lambda: (w * 3).item(),          # 6
}
for name, fn in attempts.items():
    try:
        print(f"{name:22s} -> {fn()!r}")
    except Exception as e:
        print(f"{name:22s} -> {type(e).__name__}: {str(e)[:80]}")

out.item()             -> 9.0
out.numpy()            -> RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() ins
out.detach().numpy()   -> array(9., dtype=float32)
float(out)             -> 9.0
(w * 3).tolist()       -> [3.0, 6.0]
(w * 3).item()         -> RuntimeError: a Tensor with 2 elements cannot be converted to Scalar


/tmp/ipykernel_3546/2641773512.py:8: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  "float(out)":            lambda: float(out),              # 4


Which two failed, and why? When you log `loss` values in a training loop, which form should you append to your list — `loss` or `loss.item()` — and why?

**Answer.** `out.numpy()` fails because the tensor requires grad (it is attached to the graph); you must `.detach()` first. `(w * 3).item()` fails because `.item()` only works for a tensor with exactly **one** element. In a training loop append `loss.item()` (a plain Python float): it is what `plt.plot` and `np.mean` expect, and it does not keep the whole computation graph of that batch alive in memory. (The lecturer's notebook appends the tensor and later wraps the list in `torch.tensor(...)`, which also works but keeps the graphs until then.)

---
## Blank-page drill

No scaffolding. In the empty cell below, **from memory**, write code that

1. creates a NumPy array `X` of shape `(10, 4)` with random normal values and a label vector `y` of 10 random integers in `{0, 1, 2}` (use `np.random.default_rng(0)`),
2. converts both to tensors with the dtypes a **multi-class** network and loss need,
3. pushes `X` through `nn.Linear(4, 3)` and prints the output shape,
4. computes the accuracy of the (untrained) layer as a Python float.

Then modify step 2 as if the task were **regression** on a float target: which dtype and shape must the label tensor have?

In [29]:
rng = np.random.default_rng(0)
X = rng.normal(size=(10, 4))
y = rng.integers(0, 3, size=10)

X_t = torch.tensor(X, dtype=torch.float32)       # (10, 4) float32
y_t = torch.tensor(y, dtype=torch.int64)         # (10,)   int64  -> class indices for CrossEntropyLoss

lin = nn.Linear(4, 3)
out = lin(X_t)
print(out.shape)                                 # torch.Size([10, 3])
print((out.argmax(dim=1) == y_t).float().mean().item())

# regression variant: float32 and the same shape as the model output, (10, 1)
y_reg = torch.tensor(rng.normal(size=10), dtype=torch.float32).reshape(-1, 1)
print(y_reg.shape, y_reg.dtype)

torch.Size([10, 3])
0.4000000059604645
torch.Size([10, 1]) torch.float32
